In [7]:
!pip install torchvision

  Using cached torchvision-0.26.0-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
Using cached torchvision-0.26.0-cp311-cp311-manylinux_2_28_x86_64.whl (7.5 MB)

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
# Imports

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
from torchvision import transforms
import torchvision.models as models
import zipfile
from PIL import Image
import io
from tqdm import tqdm


In [3]:
# Preprocessing images

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


# Load pretrained VGG16
backbone = models.vgg16(pretrained=True)

# Keep only the convolutional feature layers
feature_extractor = backbone.features

feature_extractor.eval()

/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (17): Conv2d(256, 512, kernel_si

In [4]:
# Feature extraction

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # good to confirm what you're actually on

# Move model to device
feature_extractor = feature_extractor.to(device)


# ------Save the features from train_images------

zip_path_train = "/home/jovyan/megan/labb3/data/coco/train_images.zip"

feature_dict_train = {}

with zipfile.ZipFile(zip_path_train, 'r') as zf:
    image_files_train = [f for f in zf.namelist() if f.endswith('.jpg')]

    for fname in tqdm(image_files_train, desc="Extracting train features"):
        with zf.open(fname) as f:
            img = Image.open(io.BytesIO(f.read())).convert("RGB")
            img_tensor = transform(img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                features = feature_extractor(img_tensor) # (1, 512, 7, 7)
            
            feature_dict_train[fname] = features.view(features.size(0), -1).squeeze().cpu() # (25088,)
            

torch.save(feature_dict_train, "/home/jovyan/annabell/train_features.pt")


# ------Save the features from val_images------

zip_path_val = "/home/jovyan/megan/labb3/data/coco/val_images.zip"

feature_dict_val = {}

with zipfile.ZipFile(zip_path_val, 'r') as zf:
    image_files_val = [f for f in zf.namelist() if f.endswith('.jpg')]
    
    for fname in tqdm(image_files_val, desc="Extracting val features"):
        with zf.open(fname) as f:
            img = Image.open(io.BytesIO(f.read())).convert("RGB")
            img_tensor = transform(img).unsqueeze(0).to(device)
            
            with torch.no_grad():
                features = feature_extractor(img_tensor) # (1, 512, 7, 7)
            
            feature_dict_val[fname] = features.view(features.size(0), -1).squeeze().cpu() # (25088,)
            

torch.save(feature_dict_val, "/home/jovyan/annabell/val_features.pt")
    

Using device: cuda


Extracting val features: 100%|██████████| 5000/5000 [00:57<00:00, 87.24it/s]


['train2017/000000147328.jpg', 'train2017/000000414738.jpg', 'train2017/000000281563.jpg', 'train2017/000000063879.jpg', 'train2017/000000531349.jpg']
